##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.functions import sum as spark_sum_val, countDistinct, collect_set, max as spark_max, round as spark_round, current_timestamp


### Payments Table Data Manipulation and Cleaning


In [0]:
df_payments_bronze = spark.table("olist_ecommerce_project.bronze.brz_payments")

# Basic profiling
print("Total rows:", df_payments_bronze.count())
print("Distinct order_id:", df_payments_bronze.select("order_id").distinct().count())

# Null check
df_payments_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_payments_bronze.columns
]).show()

# Check distinct payment types
print("\nDistinct payment types:")
df_payments_bronze.groupBy("payment_type").count().orderBy("count", ascending=False).show(truncate=False)

# Check for invalid payment values
print("Zero or negative payment_value rows:")
df_payments_bronze.filter(col("payment_value") <= 0).count()

# Understand the multiple payments per order situation
print("\nPayment sequential breakdown:")
df_payments_bronze.groupBy("payment_sequential").count().orderBy("payment_sequential").show(truncate=False)

In [0]:
# Check 1: Which order is missing from payments?
df_orders_silver = spark.table("olist_ecommerce_project.silver.slv_orders")

missing_payment = df_orders_silver.join(
    df_payments_bronze.select("order_id").distinct(),
    on="order_id",
    how="left_anti"
)
print("Orders with no payment record:")
missing_payment.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp"
).show(truncate=False)

# Check 2: Inspect the 3 'not_defined' payment type rows
print("\nnot_defined payment rows:")
df_payments_bronze.filter(
    col("payment_type") == "not_defined"
).show(truncate=False)

This code identifies two issues: it flags one delivered order with no payment record as a missing payment, and it drops three meaningless rows where both payment type and value are undefined.

In [0]:


# Step 1: Drop the 3 'not_defined' rows
df_payments_clean = df_payments_bronze.filter(
    col("payment_type") != "not_defined"
)

print("Rows after dropping not_defined:", df_payments_clean.count())

# Step 2: Referential integrity check
orphan_payments = df_payments_clean.join(
    df_orders_silver.select("order_id"),
    on="order_id",
    how="left_anti"
)
print("Payments with no matching order:", orphan_payments.count())

# Step 3: Aggregate multiple payment rows per order into one summary row
df_payments_silver = (
    df_payments_clean
    .groupBy("order_id")
    .agg(
        spark_sum_val("payment_value").alias("total_payment_value"),
        spark_max("payment_installments").alias("max_payment_installments"),
        countDistinct("payment_type").alias("payment_methods_used"),
        collect_set("payment_type").alias("payment_types")
    )
)

# Step 4: Round total_payment_value to avoid floating point issues
df_payments_silver = df_payments_silver.withColumn(
    "total_payment_value",
    spark_round(col("total_payment_value"), 2)
)

# Step 5: Add ingestion timestamp
df_payments_silver = df_payments_silver.withColumn(
    "_ingestion_timestamp", current_timestamp()
)

# Sanity check
print("\nFinal silver payments rows:", df_payments_silver.count())
df_payments_silver.show(10, truncate=False)

### Creating Payments Silver Table

In [0]:
(
    df_payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_payments")
)

print("slv_payments written successfully")

## Payments — Silver Layer Cleaning Notes

The Payments table records all payment transactions for each order. A single 
order can have multiple payment rows — for example, a customer paying partly 
with a voucher and partly with a credit card, or splitting a payment across 
multiple installments. This means `order_id` is not unique in this table at 
the Bronze level.

### Profiling Summary

| Metric | Value |
|---|---|
| Total rows | 103,886 |
| Distinct order_ids | 99,440 |
| Null values | None |
| Zero/negative payment values | None |
| Distinct payment types | 5 (credit_card, boleto, voucher, debit_card, not_defined) |
| Max payment sequential | 20+ |

### Checks Performed

**1. Missing Order Payment**
- Found 1 order (`bfbd0f9b...`) marked as `delivered` in `slv_orders` 
  but with no corresponding payment record in the Payments table
- Decision: Cannot add a row that doesn't exist in the source data
- This gap is documented here for awareness — Gold layer queries 
  on revenue should be aware 1 delivered order has no payment record

**2. Not Defined Payment Type**
- Found 3 rows with `payment_type = not_defined` and `payment_value = 0.0`
- Unlike zero freight (which was a legitimate free shipping promotion), 
  these rows have both undefined type AND zero value — completely meaningless
- Decision: Dropped all 3 rows — keeping them would pollute payment 
  type distribution analysis in Gold

**3. Referential Integrity Check**
- Verified every `order_id` in Payments exists in `slv_orders` → 0 orphans found
- Perfect referential integrity confirmed ✅

**4. Zero/Negative Payment Values**
- No zero or negative payment values found across 103,886 rows ✅

### Transformations Applied

**1. Dropped `not_defined` rows**
- Removed 3 rows with undefined payment type and zero value

**2. Aggregated multiple payment rows per order**
- Collapsed multiple payment rows per order into one summary row
- This makes joining with Orders and Order Items straightforward in Gold
- The following aggregations were applied:

| New Column | Aggregation | Meaning |
|---|---|---|
| `total_payment_value` | SUM of all payment_value | Total amount paid for the order |
| `max_payment_installments` | MAX of payment_installments | Highest installment plan used |
| `payment_methods_used` | COUNT DISTINCT of payment_type | Number of different payment methods |
| `payment_types` | COLLECT SET of payment_type | Array of all payment types used |

**3. Rounded `total_payment_value`**
- Rounded to 2 decimal places to avoid floating point precision issues

**4. Dropped `_source_file`**
- Removed Bronze-specific audit column as per Silver layer standard

### Gold Layer Use Cases

| Column | Gold Use Case |
|---|---|
| `total_payment_value` | Total revenue per order, per seller, per state |
| `max_payment_installments` | Credit behavior analysis — do high installments correlate with late delivery? |
| `payment_methods_used` | Payment mix analysis — how many orders use multiple methods? |
| `payment_types` | Filter revenue by payment method type |